# Sliding Window Counting Patterns
## A Comprehensive Guide: atMost · Prefix Sum · At Least

---

> **Who is this for?** You already know basic sliding window (two pointers, shrink/expand). This notebook teaches you the three patterns used when you need to *count* subarrays with an exact constraint — not just find one.

---

## The Core Problem

Sliding window is easy when you want to **find** the longest/shortest subarray satisfying a condition. But when you need to **count** subarrays with *exactly k* of something, a direct approach breaks down.

**Why?** Because "exactly k" is not monotone:
- A window with exactly k distincts becomes invalid if you expand *or* shrink.
- You can't cleanly track which windows are valid.

The fix: convert `exactly(k)` into two `atMost(k)` calls — which *are* monotone.

---

## Three Patterns at a Glance

| Pattern | Use When | Count Formula |
|---|---|---|
| **atMost trick** | Exactly k, non-additive (distinct count) | `count += r - l + 1` |
| **Prefix Sum** | Exactly k, additive (sum = k) | `count += freq[prefix - k]` |
| **At Least** | At least k, or complement counting | `count += n - r` |

---

# Pattern 1 — The `atMost(k)` Trick

## The Identity

$$\text{exactly}(k) = \text{atMost}(k) - \text{atMost}(k-1)$$

**Why does this work?**

- `atMost(k)` counts subarrays with **1, 2, 3, … k** distinct elements
- `atMost(k-1)` counts subarrays with **1, 2, … k-1** distinct elements
- Subtracting removes everything *except* exactly k

## Why `count += (r - l + 1)`?

At every step where the right pointer is at index `r`, and the left pointer has been shrunk to `l` (so the window [l..r] is valid):

- ALL subarrays ending at `r` that start anywhere from `l` to `r` are valid
- That's `r - l + 1` subarrays counted in ONE line
- This works because of monotonicity: if [l..r] is valid, then [l+1..r], [l+2..r], … [r..r] are all valid too

This batch-counting is what makes the algorithm O(n) instead of O(n²).

In [ ]:
from collections import defaultdict

# ──────────────────────────────────────────────────────
# PATTERN 1: atMost(k) Template
# ──────────────────────────────────────────────────────

def at_most(arr, k):
    """
    Count subarrays with AT MOST k distinct elements.
    
    Key insight: count += (r - l + 1) counts ALL valid
    subarrays ending at r in a single operation.
    """
    freq = defaultdict(int)   # frequency of each element in window
    distinct = 0              # number of unique elements in window
    count = 0
    l = 0

    for r in range(len(arr)):
        # STEP 1: Expand — add arr[r] to window
        freq[arr[r]] += 1
        if freq[arr[r]] == 1:       # new element seen
            distinct += 1

        # STEP 2: Shrink — restore "at most k" invariant
        while distinct > k:
            freq[arr[l]] -= 1
            if freq[arr[l]] == 0:   # element fully removed
                distinct -= 1
            l += 1

        # STEP 3: Count — all subarrays ending at r, starting from l to r
        count += r - l + 1          # ← THE KEY LINE

    return count


def exactly_k(arr, k):
    """Count subarrays with EXACTLY k distinct elements."""
    return at_most(arr, k) - at_most(arr, k - 1)


# ── Test ──────────────────────────────────────────────
arr = [1, 2, 1, 2, 3]
k   = 2

print(f"Array: {arr}, k = {k}")
print(f"atMost({k})   = {at_most(arr, k)}")
print(f"atMost({k-1})   = {at_most(arr, k-1)}")
print(f"exactly({k})  = {exactly_k(arr, k)}")

In [ ]:
# ──────────────────────────────────────────────────────
# STEP-BY-STEP TRACE — see exactly what happens
# ──────────────────────────────────────────────────────

def at_most_verbose(arr, k):
    freq = defaultdict(int)
    distinct = 0
    count = 0
    l = 0

    print(f"\natMost({k}) on {arr}")
    print("-" * 65)
    print(f"{'r':>3} {'l':>3} {'window':>15} {'distinct':>9} {'added':>7} {'total':>7}")
    print("-" * 65)

    for r in range(len(arr)):
        freq[arr[r]] += 1
        if freq[arr[r]] == 1:
            distinct += 1

        while distinct > k:
            freq[arr[l]] -= 1
            if freq[arr[l]] == 0:
                distinct -= 1
            l += 1

        added = r - l + 1
        count += added
        window_str = str(arr[l:r+1])
        print(f"{r:>3} {l:>3} {window_str:>15} {distinct:>9} {added:>7} {count:>7}")

    print("-" * 65)
    print(f"Total: {count}")
    return count


arr = [1, 2, 1, 2, 3]
at_most_verbose(arr, 2)
print()
at_most_verbose(arr, 1)

In [ ]:
# ──────────────────────────────────────────────────────
# LEETCODE 992 — Subarrays with K Different Integers
# ──────────────────────────────────────────────────────
# Direct application of exactly(k) = atMost(k) - atMost(k-1)

def subarrays_with_k_distinct(nums, k):
    """
    LC 992 — Hard
    Given an integer array nums and an integer k,
    return the number of good subarrays (subarrays with exactly k distinct integers).
    """
    def at_most(k):
        freq = defaultdict(int)
        distinct = 0
        count = 0
        l = 0
        for r in range(len(nums)):
            freq[nums[r]] += 1
            if freq[nums[r]] == 1:
                distinct += 1
            while distinct > k:
                freq[nums[l]] -= 1
                if freq[nums[l]] == 0:
                    distinct -= 1
                l += 1
            count += r - l + 1
        return count

    return at_most(k) - at_most(k - 1)


# Test cases
print(subarrays_with_k_distinct([1,2,1,3,4], 3))  # Expected: 3
print(subarrays_with_k_distinct([1,2,1,2,3], 2))  # Expected: 7
print(subarrays_with_k_distinct([1,1,1,1,1], 1))  # Expected: 15

In [ ]:
# ──────────────────────────────────────────────────────
# LEETCODE 1248 — Count Number of Nice Subarrays
# ──────────────────────────────────────────────────────
# Exactly k odd numbers → map odds to 1, evens to 0
# then it becomes "exactly k ones" → same atMost pattern!

def number_of_subarrays(nums, k):
    """
    LC 1248 — Medium
    A "nice" subarray has exactly k odd numbers.
    
    Trick: track the COUNT of odd numbers instead of distinct elements.
    The at_most structure is identical — just swap the constraint.
    """
    def at_most_odds(k):
        odds = 0      # count of odd numbers in window (replaces 'distinct')
        count = 0
        l = 0
        for r in range(len(nums)):
            if nums[r] % 2 == 1:    # expand: is it odd?
                odds += 1
            while odds > k:         # shrink: too many odds
                if nums[l] % 2 == 1:
                    odds -= 1
                l += 1
            count += r - l + 1      # same key line!
        return count

    return at_most_odds(k) - at_most_odds(k - 1)


print(number_of_subarrays([1,1,2,1,1], 3))  # Expected: 2
print(number_of_subarrays([2,4,6],     1))  # Expected: 0
print(number_of_subarrays([2,2,2,1,2,2,1,2,2,2], 2))  # Expected: 16

---

# Pattern 2 — Prefix Sum

## When to use it

Use prefix sum when the constraint is **additive** (e.g. sum of subarray = k). The key property: `sum(l..r) = prefix[r+1] - prefix[l]`. So finding subarrays with sum = k becomes finding pairs of prefix values with a fixed difference.

## The Identity

$$\text{sum}(l..r) = k \iff \text{prefix}[r+1] - \text{prefix}[l] = k \iff \text{prefix}[l] = \text{prefix}[r+1] - k$$

So as you walk `r` across the array, you just ask: **how many previous prefix values equal `current_prefix - k`?**

## Why NOT use atMost here?

For sums, `atMost(k)` means sum <= k, which requires non-negative numbers to work properly. Prefix sum handles negative numbers and arbitrary sums cleanly — it's a better fit for additive constraints.

In [ ]:
# ──────────────────────────────────────────────────────
# PATTERN 2: Prefix Sum Template
# ──────────────────────────────────────────────────────

def subarray_sum_equals_k(nums, k):
    """
    Count subarrays with sum EXACTLY equal to k.
    
    Core idea:
        sum(l..r) = prefix[r+1] - prefix[l]
        We want this = k, so prefix[l] = prefix[r+1] - k
    
    We use a hashmap to store: {prefix_value: how_many_times_seen}
    As we scan right, we check how many past prefixes equal (current - k).
    """
    prefix_count = defaultdict(int)
    prefix_count[0] = 1   # empty prefix (before index 0) has sum 0

    prefix = 0
    count = 0

    for r in range(len(nums)):
        prefix += nums[r]                       # extend prefix to r
        count += prefix_count[prefix - k]       # how many valid left endpoints?
        prefix_count[prefix] += 1               # record this prefix for future rs

    return count


# Test cases — works with negative numbers too!
print(subarray_sum_equals_k([1, 1, 1],     2))   # Expected: 2
print(subarray_sum_equals_k([1, 2, 3],     3))   # Expected: 2
print(subarray_sum_equals_k([-1,-1,1],     0))   # Expected: 1 — note: negatives work!
print(subarray_sum_equals_k([1,-1,0,1,2], 2))   # Expected: 4

In [ ]:
# ──────────────────────────────────────────────────────
# PREFIX SUM — Verbose trace so you can SEE it
# ──────────────────────────────────────────────────────

def prefix_sum_verbose(nums, k):
    prefix_count = defaultdict(int)
    prefix_count[0] = 1
    prefix = 0
    count = 0

    print(f"\nPrefix Sum on {nums}, k={k}")
    print("-" * 72)
    print(f"{'r':>3} {'nums[r]':>8} {'prefix':>8} {'need':>6} {'found':>6} {'total':>7}")
    print("-" * 72)

    for r in range(len(nums)):
        prefix += nums[r]
        need   = prefix - k
        found  = prefix_count[need]
        count += found
        print(f"{r:>3} {nums[r]:>8} {prefix:>8} {need:>6} {found:>6} {count:>7}")
        prefix_count[prefix] += 1

    print("-" * 72)
    print(f"Answer: {count}")
    return count


prefix_sum_verbose([1, 1, 1], 2)

In [ ]:
# ──────────────────────────────────────────────────────
# LEETCODE 930 — Binary Subarrays With Sum
# ──────────────────────────────────────────────────────
# Binary array → sum = count of 1s → prefix sum is perfect
# (Could also use atMost since values are non-negative —
#  prefix sum is cleaner here)

def num_subarrays_with_sum(nums, goal):
    """
    LC 930 — Medium
    Binary array (0s and 1s), count subarrays with sum = goal.
    """
    prefix_count = defaultdict(int)
    prefix_count[0] = 1
    prefix = 0
    count = 0
    for num in nums:
        prefix += num
        count += prefix_count[prefix - goal]
        prefix_count[prefix] += 1
    return count


print(num_subarrays_with_sum([1,0,1,0,1], 2))  # Expected: 4
print(num_subarrays_with_sum([0,0,0,0,0], 0))  # Expected: 15
print(num_subarrays_with_sum([1,1,1],     2))  # Expected: 2

In [ ]:
# ──────────────────────────────────────────────────────
# LEETCODE 560 — Subarray Sum Equals K
# ──────────────────────────────────────────────────────

def subarray_sum_lc560(nums, k):
    """
    LC 560 — Medium
    Given integer array (can have negatives), count subarrays summing to k.
    
    Must use prefix sum — atMost won't work with negative numbers
    because the window isn't monotone when elements can be negative.
    """
    prefix_count = defaultdict(int)
    prefix_count[0] = 1
    prefix = 0
    count = 0
    for num in nums:
        prefix += num
        count += prefix_count[prefix - k]
        prefix_count[prefix] += 1
    return count


print(subarray_sum_lc560([1, 2, 3],     3))  # Expected: 2 → [1,2] and [3]
print(subarray_sum_lc560([1, -1, 1],    1))  # Expected: 3 — negatives handled!
print(subarray_sum_lc560([3, 4, 7, 2, -3, 1, 4, 2], 7))  # Expected: 4

---

# Pattern 3 — At Least

## Two sub-patterns

### 3A. Direct `count += n - r`

When you have a valid left pointer `l` and you want to count subarrays starting at `l` that satisfy the condition, every right endpoint from `r` to `n-1` works. That's `n - r` subarrays — counted in one line.

This is the **mirror** of the atMost pattern:
- `atMost`: fix `r`, count all valid starting points → `r - l + 1`
- `atLeast`: fix `l`, count all valid ending points → `n - r`

### 3B. Complement counting

$$\text{at least k} = \text{total} - \text{at most (k-1)}$$

Total subarrays of length n = `n*(n+1)//2`. Subtract those with fewer than k.

In [ ]:
# ──────────────────────────────────────────────────────
# PATTERN 3A: At Least — direct count
# ──────────────────────────────────────────────────────

def count_at_least_k_distinct_3a(arr, k):
    """
    Count subarrays with AT LEAST k distinct elements.
    
    Strategy: for each valid left pointer l (where window has >= k distinct),
    every extension to the right is also valid.
    So count += (n - r) for each valid starting position.
    
    We fix the LEFT pointer and count valid RIGHT endpoints.
    (Mirror of atMost which fixes r and counts valid l)
    """
    n = len(arr)
    freq = defaultdict(int)
    distinct = 0
    count = 0
    l = 0

    for r in range(n):
        # Expand right
        freq[arr[r]] += 1
        if freq[arr[r]] == 1:
            distinct += 1

        # Once we have >= k distinct, shrink from left
        # while still maintaining >= k distinct
        while distinct >= k:
            # Every extension of this window to the right is valid
            count += n - r          # ← THE KEY LINE (mirror of r-l+1)
            freq[arr[l]] -= 1
            if freq[arr[l]] == 0:
                distinct -= 1
            l += 1

    return count


# ──────────────────────────────────────────────────────
# PATTERN 3B: At Least — complement counting (simpler)
# ──────────────────────────────────────────────────────

def count_at_least_k_distinct_3b(arr, k):
    """
    Complement approach:
        at_least(k) = total_subarrays - at_most(k-1)
    
    Often simpler and cleaner than the direct approach.
    """
    n = len(arr)
    total = n * (n + 1) // 2
    return total - at_most(arr, k - 1)


# Test both approaches — they must agree
test_cases = [
    ([1, 2, 1, 3, 4], 2),
    ([1, 2, 3],       2),
    ([1, 1, 1, 1],    1),
]

print("Verifying both at-least approaches agree:")
print(f"{'Array':>25} {'k':>3} {'3A':>6} {'3B':>6} {'Match':>7}")
print("-" * 50)
for arr, k in test_cases:
    a = count_at_least_k_distinct_3a(arr, k)
    b = count_at_least_k_distinct_3b(arr, k)
    print(f"{str(arr):>25} {k:>3} {a:>6} {b:>6} {'✓' if a==b else '✗':>7}")

In [ ]:
# ──────────────────────────────────────────────────────
# LEETCODE 2799 — Count Complete Subarrays in an Array
# ──────────────────────────────────────────────────────
# A subarray is "complete" if it contains ALL distinct elements from the array.
# This is an "at least total_distinct" problem.

def count_complete_subarrays(nums):
    """
    LC 2799 — Medium
    A complete subarray contains all distinct elements from nums.
    
    Strategy: total distinct = k. We need subarrays with AT LEAST k distinct.
    Use complement: total - atMost(k-1).
    """
    k = len(set(nums))            # total distinct elements in array
    n = len(nums)
    total = n * (n + 1) // 2
    return total - at_most(nums, k - 1)


print(count_complete_subarrays([1,3,1,2,2]))  # Expected: 4
print(count_complete_subarrays([5,5,5,5]))    # Expected: 10

In [ ]:
# ──────────────────────────────────────────────────────
# AT LEAST — Verbose trace (3A approach)
# ──────────────────────────────────────────────────────

def at_least_verbose(arr, k):
    n = len(arr)
    freq = defaultdict(int)
    distinct = 0
    count = 0
    l = 0

    print(f"\natLeast({k}) on {arr}")
    print("-" * 70)
    print(f"{'r':>3} {'l':>3} {'window':>15} {'distinct':>9} {'added':>10} {'total':>7}")
    print("-" * 70)

    for r in range(n):
        freq[arr[r]] += 1
        if freq[arr[r]] == 1:
            distinct += 1

        step_added = 0
        snap_l = l
        while distinct >= k:
            step_added += (n - r)
            count += (n - r)
            freq[arr[l]] -= 1
            if freq[arr[l]] == 0:
                distinct -= 1
            l += 1

        window_str = str(arr[snap_l:r+1])
        print(f"{r:>3} {snap_l:>3} {window_str:>15} {len(set(arr[snap_l:r+1])):>9} {step_added:>10} {count:>7}")

    print("-" * 70)
    print(f"Total: {count}")
    return count


at_least_verbose([1, 2, 1, 3, 4], 2)

---

# Pattern Comparison & Decision Guide

## The Three Counting Formulas — Side by Side

```
atMost (fix r, count lefts):
  count += r - l + 1
  "all subarrays ending at r, starting anywhere from l to r"

atLeast (fix r, count rights):
  count += n - r
  "all subarrays starting at l, ending anywhere from r to n-1"

prefix sum (fix prefix[r], count matching past prefixes):
  count += freq[prefix - k]
  "all left endpoints l where sum(l..r) = k"
```

## Decision Flowchart

```
Q: Does the constraint involve NEGATIVE numbers?
├── YES → Use PREFIX SUM  (sliding window breaks with negatives)
└── NO  →
     Q: Is the constraint ADDITIVE (sum, count of a value)?
     ├── YES → Either PREFIX SUM or atMost trick works.
     │         Prefix sum is often cleaner for pure sums.
     └── NO  → It's non-additive (distinct count, frequency-based)
               Use the atMost TRICK.

Q: Is the target "at least k" instead of "exactly k"?
├── Use complement: total - atMost(k-1)  [usually cleanest]
└── Or use the direct atLeast pattern: count += n - r
```

In [ ]:
# ──────────────────────────────────────────────────────
# SIDE-BY-SIDE: Same problem, both atMost and prefix sum
# (Binary array: both work — compare them)
# ──────────────────────────────────────────────────────

def binary_subarrays_with_sum_atmost(nums, goal):
    """atMost approach — works because values are non-negative."""
    def at_most_sum(k):
        if k < 0:
            return 0
        l, count, window_sum = 0, 0, 0
        for r in range(len(nums)):
            window_sum += nums[r]
            while window_sum > k:
                window_sum -= nums[l]
                l += 1
            count += r - l + 1
        return count
    return at_most_sum(goal) - at_most_sum(goal - 1)


def binary_subarrays_with_sum_prefix(nums, goal):
    """Prefix sum approach — handles negatives too."""
    prefix_count = defaultdict(int)
    prefix_count[0] = 1
    prefix = 0
    count = 0
    for num in nums:
        prefix += num
        count += prefix_count[prefix - goal]
        prefix_count[prefix] += 1
    return count


test = [1, 0, 1, 0, 1]
goal = 2
print(f"Input: {test}, goal={goal}")
print(f"atMost approach:  {binary_subarrays_with_sum_atmost(test, goal)}")
print(f"Prefix approach:  {binary_subarrays_with_sum_prefix(test, goal)}")
print("Both should be 4 ✓" if binary_subarrays_with_sum_atmost(test, goal) == 4 else "Mismatch!")

---

# Common Mistakes & How to Avoid Them

---

### Mistake 1: Trying to count "exactly k" with a single direct pass

```python
# WRONG — don't do this
for r in range(n):
    add arr[r]
    while distinct > k:   # shrinking when > k
        remove arr[l]; l++
    if distinct == k:     # but == k check misses windows where
        count += r-l+1    # shrinking pushed l too far right
```

**Why it fails:** When `distinct == k+1`, you shrink until `distinct == k`. But you may overshoot — now `distinct < k` and the window that had exactly k is lost.

**Fix:** Use `exactly(k) = atMost(k) - atMost(k-1)`.

---

### Mistake 2: Wrong shrink condition

```python
# WRONG
while distinct >= k:   # this shrinks until distinct < k
    ...

# CORRECT for atMost(k)
while distinct > k:    # shrink only when OVER the limit
    ...
```

---

### Mistake 3: Forgetting `prefix_count[0] = 1`

```python
# WRONG — misses subarrays starting from index 0
prefix_count = defaultdict(int)  # no initialization

# CORRECT
prefix_count = defaultdict(int)
prefix_count[0] = 1              # empty prefix (before element 0) has sum 0
```

This handles subarrays like `[0..r]` where the entire prefix sums to k.

---

### Mistake 4: Using atMost with negative numbers

```python
# WRONG — atMost breaks with negatives
# Because shrinking the window can INCREASE the sum (if you remove a negative)
# so the window is no longer monotone.

# CORRECT — use prefix sum for negative numbers
```

---

### Mistake 5: Off-by-one in `at_most(arr, k-1)`

When computing `exactly(k) = atMost(k) - atMost(k-1)`, some people write `atMost(k) - atMost(k)` by accident, or use `k-1` in the wrong place. Always verify with a small example.

In [ ]:
# ──────────────────────────────────────────────────────
# VERIFICATION SUITE — test all three patterns
# ──────────────────────────────────────────────────────

def brute_force_exactly_k_distinct(arr, k):
    """O(n²) brute force to verify atMost trick."""
    n = len(arr)
    count = 0
    for i in range(n):
        for j in range(i, n):
            if len(set(arr[i:j+1])) == k:
                count += 1
    return count


def brute_force_sum_equals_k(arr, k):
    """O(n²) brute force to verify prefix sum."""
    n = len(arr)
    count = 0
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += arr[j]
            if s == k:
                count += 1
    return count


def brute_force_at_least_k_distinct(arr, k):
    """O(n²) brute force to verify atLeast."""
    n = len(arr)
    count = 0
    for i in range(n):
        for j in range(i, n):
            if len(set(arr[i:j+1])) >= k:
                count += 1
    return count


import random
random.seed(42)

print("Verifying all patterns against brute force (10 random tests each)\n")

all_pass = True

for _ in range(10):
    arr = [random.randint(1, 4) for _ in range(random.randint(4, 10))]
    k   = random.randint(1, 3)

    # atMost
    expected = brute_force_exactly_k_distinct(arr, k)
    got      = exactly_k(arr, k)
    if expected != got:
        print(f"FAIL atMost: arr={arr} k={k} expected={expected} got={got}")
        all_pass = False

    # prefix sum
    arr2     = [random.randint(-2, 4) for _ in range(random.randint(4, 10))]
    k2       = random.randint(0, 5)
    expected = brute_force_sum_equals_k(arr2, k2)
    got      = subarray_sum_equals_k(arr2, k2)
    if expected != got:
        print(f"FAIL prefix: arr={arr2} k={k2} expected={expected} got={got}")
        all_pass = False

    # atLeast complement
    expected = brute_force_at_least_k_distinct(arr, k)
    got      = count_at_least_k_distinct_3b(arr, k)
    if expected != got:
        print(f"FAIL atLeast: arr={arr} k={k} expected={expected} got={got}")
        all_pass = False

if all_pass:
    print("All 30 tests passed! ✓")

---

# Quick Reference — All Three Templates

---

## Template 1: atMost trick (exactly k, non-additive)

```python
def exactly_k(arr, k):
    return at_most(arr, k) - at_most(arr, k - 1)

def at_most(arr, k):
    freq = defaultdict(int)
    distinct = 0      # ← your constraint metric here
    count = 0
    l = 0
    for r in range(len(arr)):
        freq[arr[r]] += 1
        if freq[arr[r]] == 1: distinct += 1   # expand
        while distinct > k:                    # shrink
            freq[arr[l]] -= 1
            if freq[arr[l]] == 0: distinct -= 1
            l += 1
        count += r - l + 1                     # count
    return count
```

---

## Template 2: Prefix sum (exactly k, additive / handles negatives)

```python
def subarray_sum_equals_k(nums, k):
    prefix_count = defaultdict(int)
    prefix_count[0] = 1   # ← don't forget this!
    prefix = 0
    count = 0
    for num in nums:
        prefix += num
        count += prefix_count[prefix - k]   # count
        prefix_count[prefix] += 1
    return count
```

---

## Template 3: At least (complement approach — usually simplest)

```python
def at_least_k(arr, k):
    n = len(arr)
    total = n * (n + 1) // 2
    return total - at_most(arr, k - 1)   # reuse atMost template
```

---

## LeetCode Problem Map

| # | Problem | Pattern | Key metric |
|---|---|---|---|
| 992 | Subarrays with K Different Integers | atMost | distinct count |
| 1248 | Count Number of Nice Subarrays | atMost | count of odds |
| 560 | Subarray Sum Equals K | Prefix Sum | sum (has negatives) |
| 930 | Binary Subarrays With Sum | Prefix Sum or atMost | sum of 0/1 array |
| 2799 | Count Complete Subarrays | atLeast (complement) | distinct == total |
| 1358 | Number of Substrings with All 3 Chars | atLeast (complement) | all 3 present |

---

**The mental shortcut:**
- Got negatives? → **Prefix sum**  
- Counting distinct / frequency-based? → **atMost trick**  
- Need "at least"? → **total - atMost(k-1)**